# ScenePatch — Gemma 4 E2B reproducibility notebook

This notebook is the primary executable Gemma path for ScenePatch: creator-owned before/after media, a spoken intent, the official `google/gemma-4-E2B-it` checkpoint, three native function declarations, and a deterministic validator that never dynamically executes a model-selected function.

**Why the notebook is primary:** on 31 July 2026, the pinned browser q4f16 runtime loaded and emitted native calls on the release M4 Pro, but it proposed a commit for the controlled missing-blue-marker fixture. That fails ScenePatch's release gate. The public Pages UI must therefore remain a clearly labeled scripted fixture replay unless a later tagged browser build passes the complete gate.

**No benchmark result is embedded here.** The notebook records only values produced by the current execution. Before running it publicly, complete `docs/FIXTURE_RIGHTS.md` for the exact input hashes.

### Kaggle setup

1. Enable a GPU accelerator and internet for the initial public-model download. The official Gemma function-calling guide targets a T4 GPU.
2. Attach a private dataset named `scenepatch-owned-fixture` containing `scene-before.jpg`, `scene-after.jpg`, and `intent.wav`. Make it public only after the human rights gate is signed.
3. Run all cells. The full checkpoint is large, so this notebook is a transparent reproducibility path rather than a latency claim about the browser build.

References: [Gemma 4 E2B IT model card](https://huggingface.co/google/gemma-4-E2B-it) and [Gemma 4 function calling](https://ai.google.dev/gemma/docs/capabilities/text/function-calling-gemma4).

In [ ]:
%pip install -q -U "transformers>=5.10.1" accelerate librosa soundfile pillow jsonschema

In [ ]:
from __future__ import annotations

from collections.abc import Mapping
from pathlib import Path
import hashlib
import json
import platform
import re
import time
import unicodedata

import librosa
import soundfile as sf
import torch
import transformers
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import Audio, display
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
MODEL_REVISION = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"
MODEL_IDENTIFIER = f"{MODEL_ID}@{MODEL_REVISION}"
MAX_AUDIO_SECONDS = 10.0
SAMPLE_RATE = 16_000

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before loading the full checkpoint."
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "model_id": MODEL_IDENTIFIER,
})

## Native tool declarations and fail-closed validator

Gemma proposes calls; the code below only validates data. It does not use `eval`, `globals()`, imports, shell commands, network calls, or any function name supplied by the model. A clean proposal remains pending until a human confirms it in the application.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "record_change",
            "description": "Record one visible scene change relative to the spoken intent.",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {
                        "type": "string",
                        "description": "Short, observable description of the change.",
                    },
                    "classification": {
                        "type": "string",
                        "enum": ["intended", "unexplained", "uncertain"],
                        "description": "How this change relates to the spoken intent.",
                    },
                },
                "required": ["description", "classification"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "commit_patch",
            "description": "Propose a patch only when every visible change is intended.",
            "parameters": {
                "type": "object",
                "properties": {"summary": {"type": "string"}},
                "required": ["summary"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "block_commit",
            "description": "Propose a block when any change is unexplained or uncertain.",
            "parameters": {
                "type": "object",
                "properties": {"reason": {"type": "string"}},
                "required": ["reason"],
                "additionalProperties": False,
            },
        },
    },
]

ALLOWED_NAMES = {"record_change", "commit_patch", "block_commit"}
CLASSIFICATIONS = {"intended", "unexplained", "uncertain"}


def _object_arguments(value):
    if isinstance(value, Mapping):
        return dict(value)
    if isinstance(value, str):
        decoded = json.loads(value)
        if isinstance(decoded, Mapping):
            return dict(decoded)
    raise ValueError("Tool arguments must be a JSON object.")


def normalize_tool_calls(parsed_response):
    if not isinstance(parsed_response, Mapping):
        raise ValueError("Parsed response must be an object.")
    raw_calls = parsed_response.get("tool_calls")
    if not isinstance(raw_calls, list):
        raise ValueError("Response must contain a tool_calls list.")

    normalized = []
    for item in raw_calls:
        if not isinstance(item, Mapping):
            raise ValueError("Each tool call must be an object.")
        payload = item.get("function", item)
        if not isinstance(payload, Mapping):
            raise ValueError("Each function payload must be an object.")
        name = payload.get("name")
        if name not in ALLOWED_NAMES:
            raise ValueError(f"Unknown tool: {name!r}")
        normalized.append({"name": name, "arguments": _object_arguments(payload.get("arguments"))})
    return normalized


def _bounded_text(value, field, maximum):
    if not isinstance(value, str):
        raise ValueError(f"{field} must be a string.")
    cleaned = " ".join(value.split())
    if not 1 <= len(cleaned) <= maximum:
        raise ValueError(f"{field} must contain 1 to {maximum} characters.")
    return cleaned


def validate_and_decide(parsed_response):
    calls = normalize_tool_calls(parsed_response)
    if len(calls) > 7:
        raise ValueError("At most six changes and one terminal call are allowed.")

    changes, terminals, seen = [], [], set()
    for call in calls:
        name, arguments = call["name"], call["arguments"]
        if name == "record_change":
            if set(arguments) != {"description", "classification"}:
                raise ValueError("record_change has missing or extra arguments.")
            description = _bounded_text(arguments["description"], "description", 280)
            classification = arguments["classification"]
            if classification not in CLASSIFICATIONS:
                raise ValueError(f"Invalid classification: {classification!r}")
            duplicate_key = re.sub(r"[.!?]+$", "", unicodedata.normalize("NFKC", description)).casefold()
            if duplicate_key in seen:
                raise ValueError("Duplicate change description.")
            seen.add(duplicate_key)
            changes.append({"description": description, "classification": classification})
        else:
            expected = {"summary"} if name == "commit_patch" else {"reason"}
            if set(arguments) != expected:
                raise ValueError(f"{name} has missing or extra arguments.")
            field = next(iter(expected))
            terminals.append({"name": name, field: _bounded_text(arguments[field], field, 500)})

    if not 1 <= len(changes) <= 6:
        raise ValueError("One to six change records are required.")
    if len(terminals) != 1:
        raise ValueError("Exactly one terminal call is required.")

    unsafe = [c for c in changes if c["classification"] in {"unexplained", "uncertain"}]
    terminal = terminals[0]
    if unsafe:
        decision = "blocked_by_deterministic_policy"
    elif terminal["name"] == "block_commit":
        decision = "blocked_as_proposed"
    else:
        decision = "pending_human_confirmation"

    return {"valid": True, "changes": changes, "terminal": terminal, "decision": decision}


In [ ]:
# Deterministic policy checks; these are code tests, not model-quality results.
clean_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Red marker moved above the sketchbook", "classification": "intended"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Requested marker move only"}}},
]}
unsafe_sample = {"tool_calls": [
    {"function": {"name": "record_change", "arguments": {"description": "Blue marker is missing", "classification": "unexplained"}}},
    {"function": {"name": "commit_patch", "arguments": {"summary": "Model proposed commit"}}},
]}
assert validate_and_decide(clean_sample)["decision"] == "pending_human_confirmation"
assert validate_and_decide(unsafe_sample)["decision"] == "blocked_by_deterministic_policy"
try:
    validate_and_decide({"tool_calls": [{"function": {"name": "delete_file", "arguments": {}}}]})
except ValueError as error:
    assert "Unknown tool" in str(error)
else:
    raise AssertionError("Unknown tools must fail closed.")
print("Deterministic validator checks passed.")

## Load and normalize the human-approved fixture

The next cell intentionally stops if the exact owned fixture is absent. It does not download or substitute example media.

In [ ]:
FIXTURE_ROOT = Path("/kaggle/input/scenepatch-owned-fixture")
BEFORE_PATH = FIXTURE_ROOT / "scene-before.jpg"
AFTER_PATH = FIXTURE_ROOT / "scene-after.jpg"
AUDIO_PATH = FIXTURE_ROOT / "intent.wav"

required_files = [BEFORE_PATH, AFTER_PATH, AUDIO_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Attach the human-approved scenepatch-owned-fixture dataset. Missing: " + ", ".join(missing)
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_hashes = {path.name: sha256_file(path) for path in required_files}
print(json.dumps(input_hashes, indent=2))
print("Confirm these hashes match the signed fixture-rights record before continuing.")

In [ ]:
WORK_ROOT = Path("/kaggle/working/scenepatch")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CONTACT_SHEET = WORK_ROOT / "before-after-contact-sheet.webp"
NORMALIZED_AUDIO = WORK_ROOT / "intent-16khz-mono.wav"

def render_panel(source_path, label):
    source = ImageOps.exif_transpose(Image.open(source_path)).convert("RGB")
    scale = max(512 / source.width, 512 / source.height)
    resized = source.resize((round(source.width * scale), round(source.height * scale)), Image.Resampling.LANCZOS)
    panel = ImageOps.fit(resized, (512, 512), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, 511, 57), fill="#101412")
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 22)
    draw.text((24, 18), label, fill="#d8ff63" if label.startswith("BEFORE") else "#8ad9ff", font=font)
    return panel

sheet = Image.new("RGB", (1024, 512), "white")
sheet.paste(render_panel(BEFORE_PATH, "BEFORE · BASE"), (0, 0))
sheet.paste(render_panel(AFTER_PATH, "AFTER · WORKTREE"), (512, 0))
ImageDraw.Draw(sheet).line((512, 0, 512, 512), fill="#777777", width=2)
sheet.save(CONTACT_SHEET, format="WEBP", quality=90, method=6)

source_audio_info = sf.info(AUDIO_PATH)
if source_audio_info.duration > MAX_AUDIO_SECONDS + 0.15:
    raise ValueError(f"Keep the spoken intent under {MAX_AUDIO_SECONDS:.0f} seconds.")
waveform, _ = librosa.load(AUDIO_PATH, sr=SAMPLE_RATE, mono=True, duration=MAX_AUDIO_SECONDS)
if waveform.size == 0:
    raise ValueError("Intent audio is empty.")
sf.write(NORMALIZED_AUDIO, waveform, SAMPLE_RATE, subtype="PCM_16")

preprocess_record = {
    "contact_sheet": {"width": sheet.width, "height": sheet.height, "sha256": sha256_file(CONTACT_SHEET)},
    "audio": {
        "source_seconds": source_audio_info.duration,
        "processed_seconds": len(waveform) / SAMPLE_RATE,
        "sample_rate_hz": SAMPLE_RATE,
        "channels": 1,
        "sha256": sha256_file(NORMALIZED_AUDIO),
    },
}
print(json.dumps(preprocess_record, indent=2))
display(Image.open(CONTACT_SHEET))
display(Audio(filename=str(NORMALIZED_AUDIO)))

## Load the official checkpoint

This is the unquantized official Google checkpoint and is separate from the browser's ONNX runtime. The model repository is public and no token is embedded in this notebook.

In [ ]:
load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype="auto", device_map="auto")
model.eval()
model_load_seconds = time.perf_counter() - load_started
print({"model_id": MODEL_IDENTIFIER, "load_seconds_this_run": model_load_seconds, "device": str(model.device)})

In [ ]:
SYSTEM_PROMPT = """You are ScenePatch's visual change reviewer for a small creative desk.
Compare the labeled BEFORE and AFTER panels with the spoken intent. Silently inventory every object in BEFORE, verify that it remains present in AFTER, then check additions and location changes. Record each material visible change once, and never record unchanged objects.
Classify a requested change as intended, an unrequested change as unexplained, and ambiguous evidence as uncertain.
After recording changes, call exactly one terminal tool. Commit only if every change is intended; otherwise block.
Do not make safety, identity, theft, inventory, or forensic claims."""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image", "path": str(CONTACT_SHEET)},
            {"type": "audio", "audio": str(NORMALIZED_AUDIO)},
            {"type": "text", "text": "Review this scene change against the spoken intent and use only the declared tools."},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    tools=TOOLS,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
).to(model.device)
print({"input_keys": sorted(inputs.keys()), "input_tokens": int(inputs["input_ids"].shape[-1])})

In [ ]:
if torch.cuda.is_available():
    torch.cuda.synchronize()
inference_started = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
if torch.cuda.is_available():
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_started

input_length = inputs["input_ids"].shape[-1]
generated_tokens = generated[0][input_length:]
raw_output = processor.decode(generated_tokens, skip_special_tokens=False)
parsed_output = processor.parse_response(raw_output, prefix=inputs["input_ids"])

print("Raw Gemma generation:")
print(raw_output)
print("\nParsed response:")
print(json.dumps(parsed_output, indent=2, default=str))
print({"inference_seconds_this_run": inference_seconds, "generated_tokens": int(generated_tokens.shape[-1])})

In [ ]:
# Validation failure is intentionally loud. The production app may make one constrained repair attempt, then blocks.
audit = validate_and_decide(parsed_output)
execution_record = {
    "model_id": MODEL_IDENTIFIER,
    "model_load_seconds_this_run": model_load_seconds,
    "inference_seconds_this_run": inference_seconds,
    "input_hashes": input_hashes,
    "preprocessing": preprocess_record,
    "audit": audit,
    "human_confirmation_performed": False,
}
print(json.dumps(execution_record, indent=2))
print("Notebook complete. A pending proposal is not a commit; confirmation remains a human application action.")

## Interpreting this run

One execution demonstrates the mechanism, not model accuracy. Before publishing a performance statement, run the release protocol on the exact owned fixtures and hardware: five consecutive clean-scene proposals and five consecutive bad-scene blocks, plus the occlusion case. Report every failure and the exact environment. Do not copy this notebook's full-checkpoint timing into the browser demo; they are different runtimes.